In [1]:
import os
import cv2
import pandas as pd 
import numpy as np 


In [2]:
categories = ["COVID","Normal"]
data = []

In [11]:
data = []
categories = ["COVID", "Normal"]

for category in categories:
    label = 1 if category == "COVID" else 0
    
    # Point directly to the "images" subfolder inside COVID and Normal
    folder_path = f"./{category}/images" 
    
    if os.path.exists(folder_path):
        files = os.listdir(folder_path)
        print(f"Found {len(files)} files in {folder_path}...")
        
        for file in files:
            img_path = os.path.join(folder_path, file)
            img = cv2.imread(img_path, 0) # Read in grayscale
            
            if img is not None:
                img_resized = cv2.resize(img, (64, 64))
                data.append([img_resized.flatten(), label])
    else:
        print(f"Path does not exist: {folder_path}. Check subfolder names.")

print(f"\nSuccessfully loaded {len(data)} total images!")

Found 3616 files in ./COVID/images...
Found 10192 files in ./Normal/images...

Successfully loaded 13808 total images!


In [12]:
df = pd.DataFrame(data,columns=["pixels","label"])

In [13]:
pixel_columns = pd.DataFrame(df['pixels'].to_list(),columns=[f"pixel_{i}" for i in range(4096)])
final_df = pd.concat([pixel_columns, df["label"]],axis=1)

In [14]:
final_df.head()

,pixel_0,pixel_1,pixel_2,pixel_3,pixel_4,pixel_5,pixel_6,pixel_7,pixel_8,pixel_9,...,pixel_4087,pixel_4088,pixel_4089,pixel_4090,pixel_4091,pixel_4092,pixel_4093,pixel_4094,pixel_4095,label
0,1,0,1,0,0,0,0,0,0,0,...,171,166,163,144,103,14,0,7,5,1
1,126,124,121,119,117,114,99,99,99,97,...,198,194,192,189,184,180,172,165,161,1
2,1,0,0,0,0,0,0,0,0,0,...,235,228,211,170,104,25,0,0,1,1
3,4,4,4,4,4,3,3,3,3,3,...,163,141,82,15,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,2,...,168,168,162,156,150,143,136,118,108,1


In [15]:
final_df

,pixel_0,pixel_1,pixel_2,pixel_3,pixel_4,pixel_5,pixel_6,pixel_7,pixel_8,pixel_9,...,pixel_4087,pixel_4088,pixel_4089,pixel_4090,pixel_4091,pixel_4092,pixel_4093,pixel_4094,pixel_4095,label
0,1,0,1,0,0,0,0,0,0,0,...,171,166,163,144,103,14,0,7,5,1
1,126,124,121,119,117,114,99,99,99,97,...,198,194,192,189,184,180,172,165,161,1
2,1,0,0,0,0,0,0,0,0,0,...,235,228,211,170,104,25,0,0,1,1
3,4,4,4,4,4,3,3,3,3,3,...,163,141,82,15,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,2,...,168,168,162,156,150,143,136,118,108,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13803,0,0,0,0,0,0,0,0,0,0,...,117,82,38,1,0,0,0,0,0,0
13804,127,16,10,12,11,11,11,10,10,10,...,0,0,0,0,0,0,0,0,0,0
13805,0,0,0,0,0,0,0,0,0,0,...,232,231,229,222,221,219,216,208,199,0
13806,187,85,48,50,52,51,49,48,47,46,...,207,194,190,181,174,164,152,129,58,0


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

X = final_df.drop('label', axis=1)
y = final_df['label']

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

Features shape: (13808, 4096)
Target distribution:
label
0    10192
1     3616
Name: count, dtype: int64


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data scaled successfully!")

Data scaled successfully!


In [18]:
svm_model = SVC(kernel='rbf', C=1.0, class_weight='balanced', random_state=42)

print("Training SVM model... (this may take 1-2 minutes)")
svm_model.fit(X_train_scaled, y_train)
print("Training complete!")

Training SVM model... (this may take 1-2 minutes)
Training complete!


In [19]:
# 5. Predict on test data
y_pred = svm_model.predict(X_test_scaled)

# 6. Output metrics
print("=== Accuracy ===")
print(f"{accuracy_score(y_test, y_pred) * 100:.2f}%\n")

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Normal', 'COVID']))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

=== Accuracy ===
91.60%

=== Classification Report ===
              precision    recall  f1-score   support

      Normal       0.97      0.92      0.94      2039
       COVID       0.80      0.91      0.85       723

    accuracy                           0.92      2762
   macro avg       0.88      0.91      0.90      2762
weighted avg       0.92      0.92      0.92      2762

=== Confusion Matrix ===
[[1874  165]
 [  67  656]]


In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    HistGradientBoostingClassifier
)
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import time
import pandas as pd

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "Support Vector Machine": SVC(kernel='rbf', class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting (Hist)": HistGradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(), random_state=42, n_jobs=-1)
}

results = []

In [21]:
for name, model in models.items():
    print(f"Training {name}...")
    start_time = time.time()
    
    # Train model
    model.fit(X_train_scaled, y_train)
    train_time = time.time() - start_time
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    
    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=['Normal', 'COVID'], output_dict=True)
    
    covid_recall = report['COVID']['recall']
    covid_precision = report['COVID']['precision']
    covid_f1 = report['COVID']['f1-score']
    
    results.append({
        "Model": name,
        "Accuracy": f"{acc * 100:.2f}%",
        "COVID Recall": f"{covid_recall * 100:.2f}%",
        "COVID Precision": f"{covid_precision * 100:.2f}%",
        "COVID F1-Score": f"{covid_f1 * 100:.2f}%",
        "Train Time (s)": f"{train_time:.2f}s"
    })

Training Logistic Regression...


D:\datascience\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training Support Vector Machine...
Training Random Forest...
Training AdaBoost...
Training Gradient Boosting (Hist)...
Training XGBoost...


In [22]:
comparison_df = pd.DataFrame(results)
print("\n=== Comprehensive Model Benchmark Summary ===")
print(comparison_df.to_string(index=False))


=== Comprehensive Model Benchmark Summary ===
                   Model Accuracy COVID Recall COVID Precision COVID F1-Score Train Time (s)
     Logistic Regression   79.51%       69.71%          59.22%         64.04%         65.87s
  Support Vector Machine   91.60%       90.73%          79.90%         84.97%        217.64s
           Random Forest   91.96%       83.54%          85.43%         84.48%         23.80s
                AdaBoost   86.10%       61.83%          80.54%         69.95%        582.53s
Gradient Boosting (Hist)   94.06%       84.51%          92.16%         88.17%        103.05s
                 XGBoost   94.75%       89.49%          90.36%         89.92%         92.33s


In [23]:
import joblib

# 1. Export the winning XGBoost model
joblib.dump(models["XGBoost"], 'covid_xgboost_model.joblib')

# 2. Export the scaler
joblib.dump(scaler, 'covid_scaler.joblib')

print("Saved 'covid_xgboost_model.joblib' and 'covid_scaler.joblib' successfully!")

Saved 'covid_xgboost_model.joblib' and 'covid_scaler.joblib' successfully!
